In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

%cd /content
if os.path.exists('/content/korean-chatbot'):
    %cd korean-chatbot
    !git pull
else:
    !git clone https://github.com/kkkk2058/korean-chatbot.git
    %cd korean-chatbot

!pip install -r requirements.txt

In [ ]:
import shutil, os

os.makedirs("data", exist_ok=True)
shutil.copy("/content/drive/MyDrive/korean-chatbot/data/train.txt", "data/train.txt")

print("파일 로드 완료!")
path = "data/train.txt"
print(f"train.txt 크기: {os.path.getsize(path) / 1024 / 1024:.1f} MB")

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
import sys
sys.path.append('/content/korean-chatbot')
from src.tokenizer import load_tokenizer
from src.model import Transformer

tok = load_tokenizer()
model = Transformer()
model = model.to(device)
print(f"vocab size: {tok.vocab_size}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class TextDataset(Dataset):
    def __init__(self, path, tokenizer, max_seq_len=512):
        self.samples = []
        
        with open(path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]
        
        print(f"총 {len(lines):,}개 문장 토크나이징 중...")
        
        encoded = tokenizer(
            lines,
            truncation=True,
            max_length=max_seq_len,
            padding="max_length",
            return_tensors="pt"
        )
        
        for i in tqdm(range(len(lines)), desc="데이터셋 생성 중..."):
            self.samples.append(encoded["input_ids"][i])
        
        print(f"총 샘플 수: {len(self.samples):,}")
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

dataset = TextDataset("data/train.txt", tok)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)  # KoGPT2는 lr 작게



In [ ]:
EPOCHS = 3
for epoch in range(EPOCHS):
    total_loss = 0
    progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for step, batch in progress_bar:
        batch = batch.to(device)
        
        loss = model.loss(batch, tok.pad_token_id)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if step % 10 == 0:
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(dataloader)
    print(f"✨ Epoch {epoch+1} 완료 | 평균 loss: {avg_loss:.4f}\n")

print("🎉 모든 학습 완료!")

In [ ]:
import shutil

os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/model.pt")

os.makedirs("/content/drive/MyDrive/korean-chatbot/models", exist_ok=True)
shutil.copy("models/model.pt", "/content/drive/MyDrive/korean-chatbot/models/model.pt")
print("✅ 모델 저장 & Drive 백업 완료!")

In [ ]:
tokenizer = load_tokenizer()  # ← 추가
model.eval()                  # ← 추가


def generate_response(text, max_length=200):
    # 명확한 지시 형식으로 감싸기
    prompt = f"### 질문: {text}\n### 답변:"
    return model.generate(prompt, tokenizer, max_length=max_length)

# 테스트
test_inputs = [
    # 일상 대화
    '안녕하세요! 오늘 날씨가 어때요?',
    '요즘 어떻게 지내세요?',
    '오늘 기분이 좋지 않아요.',
    '주말에 뭐 하셨어요?',

    # 지식 질문
    '한국의 수도는 어디인가요?',
    '인공지능이 뭔지 쉽게 설명해줘.',
    '블랙홀이 뭐야?',
    '지구온난화가 왜 문제야?',

    # 코딩 관련
    '파이썬으로 리스트를 정렬하는 방법을 알려줘.',
    '재귀함수가 뭐야?',
    'for문이랑 while문 차이가 뭐야?',

    # 추천 요청
    '오늘 저녁 뭐 먹을까?',
    '심심한데 볼만한 영화 추천해줘.',
    '혼자 여행가기 좋은 국내 도시 추천해줘.',

    # 감정/고민
    '친구랑 싸웠는데 어떻게 화해해야 할까?',
    '공부가 너무 하기 싫어.',
    '요즘 잠을 못 자겠어.',
]

for prompt in test_inputs:
    print(f'입력: {prompt}')
    print(f'응답: {generate_response(prompt)}')
    print('-' * 50)

for prompt in test_inputs:
    print(f'입력: {prompt}')
    print(f'응답: {generate_response(prompt)}')
    print('-' * 50)

In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import torch, math

ds = load_dataset("beomi/KoAlpaca-v1.1a", split="train").train_test_split(test_size=0.1, seed=42)
train_data = ds['train']
val_data   = ds['test']

class KoAlpacaDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []

        for item in data:
            text = f"질문: {item['instruction']}\n답변: {item['output']}"
            tokens = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding='max_length',
                return_tensors='pt'
            )
            self.samples.append({
                'input_ids':      tokens['input_ids'].squeeze(),
                'attention_mask': tokens['attention_mask'].squeeze(),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# ↓ 클래스 밖으로 꺼냄 (들여쓰기 없음)
def evaluate(model, dataloader, device='cpu'):
    model.eval()
    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            loss = model.loss(input_ids, pad_token_id=tokenizer.pad_token_id)
            num_tokens = attention_mask.sum().item()
            total_loss += loss.item() * num_tokens
            total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity

val_dataset = KoAlpacaDataset(val_data, tokenizer)
val_loader  = DataLoader(val_dataset, batch_size=8)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
loss, ppl = evaluate(model, val_loader, device=device)
print(f'📊 Validation Loss : {loss:.4f}')
print(f'📊 Perplexity      : {ppl:.2f}')